# IDS-BGWO-SHAP — Colab launcher

Thin launcher for the *Metaheuristic FS + SHAP-in-the-loop IDS* project.
Runs top-to-bottom on Colab free tier with three interactive steps:

1. **Upload `kaggle.json`** (for the dataset download).
2. (Optional) **Paste a GitHub PAT** if you want to push results back.
3. Run the experiment matrix.

Two contributions implemented:

* **BGWO feature selection** (from scratch) replacing the baseline's information-gain/FCBF filter.
* **SHAP-in-the-loop** explanation-coherence term inside the BGWO fitness — making it tri-objective.

Baseline: Yang et al., *LCCDE* (GLOBECOM '22). Citations in the README.

## 1. Clone / pull the repo and install dependencies

In [ ]:
# Clone (or reuse) the repo and install our deps.
#
# Colab's preinstalled package soup contains 100+ libraries with
# tangled interdependencies. ANY install or uninstall here will
# trigger pip-resolver warnings about packages we don't use.
# They are noise — the actual install succeeds. We suppress
# pip's stderr entirely and print a single status line we control.
import pathlib, subprocess, sys

REPO_DIR = 'IDS-features-selection'
if not pathlib.Path(REPO_DIR).exists():
    !git clone -q https://github.com/yazanjer/IDS-features-selection.git
%cd $REPO_DIR

# Run pip install with stderr fully suppressed. We only show output
# if pip's exit code is non-zero (real failure).
print('Installing requirements (suppressing pip resolver chatter) ...')
proc = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check',
     '-r', 'requirements.txt'],
    capture_output=True, text=True,
)
if proc.returncode != 0:
    print('PIP INSTALL FAILED — full output below:')
    print(proc.stdout)
    print(proc.stderr)
    raise SystemExit(1)
print('Install OK.')

# Sanity-check the packages we actually use, ignoring everything else.
import importlib
for pkg in ('numpy', 'pandas', 'sklearn', 'scipy', 'lightgbm', 'xgboost',
            'shap', 'lime', 'imblearn', 'kagglehub'):
    m = importlib.import_module(pkg)
    print(f'  {pkg:12s} {getattr(m, "__version__", "?")}')
print('Environment ready.')

## 2. Upload your `kaggle.json` (interactive)

In [ ]:
from google.colab import files
from src.data_loader import install_kaggle_credentials_from_upload

print('Select kaggle.json from your local machine:')
uploaded = files.upload()
install_kaggle_credentials_from_upload(uploaded)

## 3. Configure the experiment
Edit the cell below to swap dataset, sample size, BGWO budget, and the alpha/beta/gamma weights.

In [ ]:
from src.config import Config

# Defaults are PAPER-GRADE: 500K stratified sample, 10 seeds,
# BGWO pop=15 / iter=30, alpha/beta/gamma = 0.85/0.10/0.05,
# inner-loop FS subset 30K/10K, SHAP bg 200.
# Override any field below; everything else inherits paper defaults.
cfg = Config(
    dataset='cicids2017',     # or 'unsw_nb15'
    # sample_size=500_000,
    # n_seeds=10,
    # bgwo_population=15, bgwo_iterations=30,
    # alpha=0.85, beta=0.10, gamma=0.05,
    # fs_train_rows=30_000, fs_test_rows=10_000,
)
cfg.ensure_dirs()
print(cfg.to_json())

## 4. Run the experiment matrix (10 seeds × 8 methods)

Baselines: full set, information-gain filter, RFE (LightGBM), L1 LASSO, RF-importance, in-house Boruta, BGWO bi-objective. Proposed: BGWO tri-objective with SHAP coherence.


In [ ]:
from src.evaluation import run_experiment_matrix, aggregate, wilcoxon_vs_reference, DEFAULT_METHODS

# DEFAULT_METHODS = ('none', 'filter', 'rfe', 'lasso', 'rf_imp', 'boruta', 'bgwo_bi', 'bgwo_shap')
df = run_experiment_matrix(cfg, methods=DEFAULT_METHODS)
agg = aggregate(df)
agg

In [ ]:
wil_f1   = wilcoxon_vs_reference(df, 'bgwo_shap', 'macro_f1')
wil_acc  = wilcoxon_vs_reference(df, 'bgwo_shap', 'accuracy')
wil_size = wilcoxon_vs_reference(df, 'bgwo_shap', 'n_features_selected')
import pandas as pd
pd.concat([wil_f1, wil_acc, wil_size], ignore_index=True)

## 5. Sensitivity sweeps (alpha/beta/gamma; BGWO pop/iters)

In [ ]:
from dataclasses import replace
from src.evaluation import run_experiment_matrix

weight_grid = [(0.9,0.05,0.05), (0.85,0.10,0.05), (0.7,0.2,0.1), (0.6,0.2,0.2)]
frames = []
for a,b,g in weight_grid:
    cfg_w = replace(cfg, alpha=a, beta=b, gamma=g, n_seeds=3)
    sub = run_experiment_matrix(cfg_w, methods=['bgwo_shap'])
    sub['alpha'], sub['beta'], sub['gamma'] = a, b, g
    frames.append(sub)
import pandas as pd
weights_df = pd.concat(frames, ignore_index=True)
weights_df.to_csv(cfg.results_dir/'sensitivity_weights.csv', index=False)
weights_df.head()

In [ ]:
pop_iter_grid = [(6,10),(10,20),(15,30)]
frames = []
for p,it in pop_iter_grid:
    cfg_pi = replace(cfg, bgwo_population=p, bgwo_iterations=it, n_seeds=3)
    sub = run_experiment_matrix(cfg_pi, methods=['bgwo_shap'])
    sub['pop'], sub['iters'] = p, it
    frames.append(sub)
pop_df = pd.concat(frames, ignore_index=True)
pop_df.to_csv(cfg.results_dir/'sensitivity_pop_iters.csv', index=False)
pop_df.head()

## 6. Plots & cross-dataset stability

In [ ]:
from src.plots import (plot_latency_vs_features, render_comparison_table,
                       plot_cross_dataset_overlap)
plot_latency_vs_features(df, cfg.results_dir)
render_comparison_table(agg, cfg.results_dir)
agg

In [ ]:
# Cross-dataset: rerun on UNSW-NB15 and overlay feature subsets.
cfg_unsw = replace(cfg, dataset='unsw_nb15', n_seeds=3)
df_unsw = run_experiment_matrix(cfg_unsw, methods=['bgwo_shap'])
df_unsw.to_csv(cfg.results_dir/'unsw_results.csv', index=False)

## 7. (Optional) Push results to GitHub

In [ ]:
import getpass, os
PAT = getpass.getpass('Paste GitHub PAT (leave blank to skip): ')
if PAT:
    REPO  = input('owner/repo (e.g. yazanal/ids-bgwo-shap): ').strip()
    EMAIL = input('git email: ').strip()
    NAME  = input('git name: ').strip()
    !git config user.email "$EMAIL"
    !git config user.name  "$NAME"
    !git add results/ && git commit -m 'Add Colab experiment results' || true
    !git remote set-url origin "https://${PAT}@github.com/${REPO}.git"
    !git push origin HEAD
else:
    print('Skipping GitHub push.')